In [0]:
from typing import Any
import logging
from py4j.java_gateway import JavaObject  
from pyspark.sql import SparkSession
from application_sme_internal import ApplicationSMEInternal
from pyspark.sql.functions import (
    coalesce,
    col,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    max,
    sha2,
    substring,
    when,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()

In [0]:
def load_curated_application_sme(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix, sme: ApplicationSMEInternal) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "ApplicationID	string,"
        #     "SourceSystem	string,"
        #     "PortfolioKey	string,"
        #     "PortfolioName	string,"
        #     "IngestionTimestamp	timestamp,"
        #     "BatchId	string,"
        #     "ApplicationKey	string,"
        #     "ApplicationStrategyKey string,"
        #     "ApplicationScheduleKey string,"
        #     "ExposureKey	string,"
        #     "AgreementKey	string,"
        #     "PartyKey	string,"
        #     "CDI	string,"
        #     "TaxRegistrationNumber string,"
        #     "GroupCode string,"
        #     "BusinessPlanKey int,"
        #     "BusinessPlanSolutionTypeCode string,"
        #     "BusinessPlanSolutionType string,"
        #     "BusinessPlanSolutionStatusCode string,"
        #     "BusinessPlanSolutionStatus string,"
        #     "BusinessPlanComments string,"
        #     "BusinessPlanCreatedDate timestamp,"
        #     "BusinessPlanCreatedBy string,"
        #     "BusinessPlanLastUpdatedDate timestamp,"
        #     "BusinessPlanLastUpdatedBy string,"
        #     "ApplicationDate timestamp,"
        #     "ApplicationTypeCode	string,"
        #     "ApplicationType	string,"
        #     "ApplicationStatusCode	string,"
        #     "ApplicationStatus	string,"
        #     "IntentionDate	date,"
        #     "ApprovedDate	timestamp,"
        #     "RejectedDate	timestamp,"
        #     "CancelledDate	timestamp,"
        #     "ImplementationDate	string,"
        #     "BeginDate	date,"
        #     "ProposedTypeCode	string,"
        #     "ProposedType	string,"
        #     "ConfidenceTypeCode	string,"
        #     "ConfidenceType	string,"
        #     "Installment	string,"
        #     "PropositionUser	string,"
        #     "Tenor	string,"
        #     "TimeToAuction	int,"
        #     "TimeToRecovery	int,"
        #     "TimeToSell	int,"
        #     "PrepaymentDate	date,"
        #     "AuctionPrice	decimal(19,2),"
        #     "BalloonAmount	decimal(19,2),"
        #     "PrepaymentAmount	decimal(19,2),"
        #     "BusinessPlanStrategySolutionTypeCode	string,"
        #     "BusinessPlanStrategySolutionType	string,"
        #     "Interest	decimal(21,4),"
        #     "InterestPaymentPeriodicity	int,"
        #     "NetRecoveryAmount	decimal(19,2),"
        #     "BusinessPlanStrategyCode	int,"
        #     "Comments	string,"
        #     "LastUpdatedDate	timestamp,"
        #     "CreatedDate	timestamp,"
        #     "CreatedBy	string,"
        #     "AssigneeUserCode	string,"
        #     "AssigneeUser	string,"
        #     "AssigneePortfolioGroupCode	string,"
        #     "AssigneePortfolioGroup	string,"
        #     "IntegrationCode	string,"
        #     "CommunicationChannelCode	string,"
        #     "CommunicationChannel	string,"
        #     "FlowSegmentCode	string,"
        #     "FlowSegment	string,"
        #     "UserCreatedQualcoGroupCode	string,"
        #     "ContractSigningMethodCode	string,"
        #     "ContractSigningMethod	string,"
        #     "ManagedByPortfolioGroupCode	string,"
        #     "InterbankFlag	boolean,"
        #     "OcwFlag	boolean,"
        #     "ContactSigningPlaceTypeCode	int,"
        #     "ContactSigningPlaceType	string,"
        #     "ReportingApplicationStatusCode	int,"
        #     "ReportingApplicationStatus	string,"
        #     "ClientResponseCode	int,"
        #     "ClientResponse	string,"
        #     "OcwSimulationVote	int,"
        #     "StartedDate	timestamp,"
        #     "SubmittedDate	timestamp,"
        #     "QualcoAppPreApprovedDate	timestamp,"
        #     "QualcoAppDisbursedDate	timestamp,"
        #     "QualcoAppFinalStepDate	timestamp,"
        #     "UserCreatedChannelCode	string,"
        #     "UserUnit	string,"
        #     "UserCreatedGroupCode	string,"
        #     "UserCreatedGroup	string,"
        #     "ApplicationImplementationStatusCode	string,"
        #     "ApplicationImplementationStatus	string,"
        #     "ApplicationActivationDate	date,"
        #     "ContactSigningPlaceCode	int,"
        #     "PolicyOverrideFlag	boolean,"
        #     "InstallmentNumber int,"
        #     "InstallmentAmount decimal(18,4),"
        #     "InstallmentDate timestamp,"
        #     "InstallmentCapitalAmount decimal(18,4),"
        #     "InstallmentChargeAmount decimal(18,4),"
        #     "InstallmentN128Amount decimal(18,4),"
        #     "InstallmentTotalChargeAmount decimal(18,4),"
        #     "InstallmentRemainingCapitalAmount decimal(18,4),"
        #     "InstallmentNPVAmount decimal(18,4),"
        #     "InstallmentNPVCalculationDate timestamp,"
        #     "InstallementStatusFlag boolean,"
        #     "IsPrimaryRowFlag int,"
        #     "BusinessDate date"
        # )
        cluster_columns = ["BusinessDate"]
        curated_table_path = f"{curated_zone}.Application"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                    spark,
                    curated_table_path,
                    run_date,  # type: ignore
                    "sme",
                )        
        else:
            logging.info("Tables doe not exist, no need to delete entries for business date!")
   
        # ================== business_plan_rawdata ==================
                    
        business_plan_rawdata_selected_fields = [
                    "BusinessPlanID",
                    "TimeToAuction",
                    "TimeToRecovery",
                    "TimeToSell",
                    "PrepaymentMonth",
                    "AuctionPrice",
                    "BalloonAmount",
                    "PrepaymentAmount",
                    "Interest",
                    "InterestPaymentPeriodicityID",
                    "NetRecoveryAmount",
                    "BP_Strategy_Code",
                    "DebtorCDI",
                    "AgreementNumber",
                    "BusinessPlan_Status",
                    "BusinessPlan_SolutionType",
                    "BusinessPlanStrategy_Status",
                    "BusinessPlanStrategy_StrategyType",
                    "BusinessPlanStrategy_SolutionType",
                    "NumberOfInstallments",
                    "ImplementationStatus", 
                    "FirstInstallmentDate", 
                    "StartingCapitalAmount", 
                    "tbl",
                    "PeriodicityID",
                    "CollectionPlan_Month", 
                    "CollectionPlan_Amount", 

                ]

        indiv_bankruptcy_stand_still = sme.businessPlanIndivBankruptcyStandStill()
        business_plan_corp_bankruptcy = sme.businessPlanCorpBankruptcy()
        tool_forc_cbs = sme.businessPlanToolForc()
        tool_vol_cbs = sme.businessPlanToolVol()
        settlement_perimeter = sme.settlementPerimeter()

        recent_indiv_bankruptcy_stand_still = sme.fetchMostRecentStrategy(source_table = indiv_bankruptcy_stand_still)
        recent_business_plan_corp_bankruptcy = sme.fetchMostRecentStrategy(source_table = business_plan_corp_bankruptcy)
        recent_tool_forc_cbs = sme.fetchMostRecentStrategy(source_table = tool_forc_cbs)
        recent_tool_vol_cbs = sme.fetchMostRecentStrategy(source_table = tool_vol_cbs)
        recent_settlement_perimeter = sme.fetchMostRecentStrategy(source_table = settlement_perimeter)

        business_plan_rawdata = (
            indiv_bankruptcy_stand_still
            .union(business_plan_corp_bankruptcy)
            .union(tool_forc_cbs)
            .union(tool_vol_cbs)
            .union(recent_settlement_perimeter)
        )
        business_plan_rawdata = business_plan_rawdata.select(business_plan_rawdata_selected_fields)

        # ================== business_plan_contract ==================

        business_plan_contract_selected_fields = ["Agreement_Number", "BusinessPlanStrategy_ID"]
        business_plan_contract = sme.businessPlanContract()
        business_plan_contract = business_plan_contract.select(business_plan_contract_selected_fields)
        business_plan_contract_distinct = business_plan_contract.distinct()

        # ================== business_plan_strategy ==================

        business_plan_strategy_selected_fields = [
                    "Comments",
                    "StrategyID", 
                    "BPSStatus", 
                    "Strategy",  
                    "LastModifiedDate",
                    "ProposedBy",
                    "Confidence",
                    "BusinessPlanID",
                    "BeginMonth",
                    "CreationUser",
                    "ID",
                    "BusinessDate",
                ]

        business_plan_strategy = sme.businessPlanStrategy()
        business_plan_strategy = (
            business_plan_strategy.withColumnRenamed("ID", "id_strategy")
                                .withColumnRenamed("BusinessPlanID", "BusinessPlanID_strategy")
                                .withColumnRenamed("StrategyID", "Strategy_id")
                                .withColumnRenamed("BPSStatus", "bps_status")
        )

        # ================== business_plan ==================

        business_plan_selected_fields = [
                    "ID",
                    "ApplicationDate",
                    "IntentionDate",
                    "DebtorCDI",
                    "DebtorAFM",
                    "GroupCode",
                    "SolutionType", 
                    "Business_plan_status", 
                    "Comments", 
                    "CreationDate", 
                    "CreationUser", 
                    "LastModifiedDate", 
                    "LastModifiedUser", 
                    "CompletionTargetDate",
                ]


        business_plan = sme.businessPlan()
        business_plan = (
                    business_plan.withColumnRenamed("LastModifiedDate", "LastmodifiedDate_bp")
                                .withColumnRenamed("CreationUser", "creationuser_bp")
                                .withColumnRenamed("Comments", "comments_bp")
                                .withColumnRenamed("CreationDate", "creationdate_bp")
                                .withColumn("DebtorCDI_prefix", substring("DebtorCDI", 1, 3))
                )

        # ================== business_plan_lbspaymentschedule ==================

        business_plan_lbspaymentschedule_selected_fields = [
                    "InstallmentSN",
                    "InstallmentTotalAmount",
                    "InstallmentDate",
                    "InstallmentCapitalAmount",
                    "InstallmentChargeAmount",
                    "InstallmentN128Amount",
                    "InstallmentTotalChargeAmount",
                    "InstallmentRemainingCapitalAmount",
                    "InstallmentNPVAmount",
                    "InstallmentNPVCalculationDate",
                    "IsActive",
                    "LBSParentID",
                    "creationdate", 
                ]

        business_plan_lbspaymentschedule = sme.lbsPaymentsSchedule()
        business_plan_lbspaymentschedule = business_plan_lbspaymentschedule.select(business_plan_lbspaymentschedule_selected_fields)

        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter

            portfolio_selected_fields = [
                    "PortfolioName",
                    "Prefix",
                    "unit_code_profits",
                ]

            portfolio = load_landed_portfolio_table( 
                    spark,
                    curated_zone, 
                    "portfolio",
                    portfolio_final_filters,
                    portfolio_selected_fields,
                )

        # ================== joins ==================

        subquery_lbs = business_plan_lbspaymentschedule.alias("A").join(
                    business_plan_lbspaymentschedule.groupBy("LBSParentID", "InstallmentSN")
                    .agg(max("creationdate").alias("creationdate"))
                    .alias("B"),
                    (col("A.LBSParentID") == col("B.LBSParentID"))
                    & (col("A.InstallmentSN") == col("B.InstallmentSN"))
                    & (col("A.creationdate") == col("B.creationdate")),
                )

        application_sme_df = (
                    business_plan_strategy.join(
                        business_plan,
                        business_plan_strategy["BusinessPlanID_strategy"] == business_plan["ID"],
                        "inner",
                    )
                    .join(
                        portfolio,
                        portfolio["Prefix"] == business_plan["DebtorCDI_prefix"],
                        "left",
                    )
                    .join(
                        business_plan_contract_distinct,
                        business_plan_strategy["id_strategy"] == business_plan_contract_distinct["BusinessPlanStrategy_ID"],
                        "left",
                    )
                    .join(
                        business_plan_rawdata,
                        (
                            coalesce(business_plan_contract_distinct["Agreement_Number"], lit("1"))
                            == coalesce(business_plan_rawdata["AgreementNumber"], lit("1"))
                        )
                        & (
                            coalesce(business_plan_strategy["Strategy"], lit("1"))
                            == coalesce(business_plan_rawdata["BusinessPlanStrategy_StrategyType"], lit("1"),)
                        )
                        & (
                            coalesce(business_plan_strategy["bps_status"], lit("1"))
                            == coalesce(business_plan_rawdata["BusinessPlanStrategy_Status"], lit("1"))
                        )
                        & (business_plan["DebtorCDI"] == business_plan_rawdata["DebtorCDI"])
                        & (business_plan_strategy["BusinessPlanID_strategy"] == business_plan_rawdata["BusinessPlanId"])
                        & (business_plan_strategy["id_strategy"] == business_plan_rawdata["BP_Strategy_Code"]),
                        "left",
                    )
                    .join(
                        subquery_lbs,
                        business_plan_strategy["BusinessPlanID_strategy"] == subquery_lbs["A.LBSParentID"],
                        "left",
                    )
                )

        application_sme_df = (
                    application_sme_df.withColumn(
                        "AgreementId",
                        coalesce(application_sme_df["Agreement_Number"], lit("-1")),
                    )
                    .withColumn(
                        "ApplicationCancelledDate",
                        when(
                            application_sme_df["bps_status"] == "Cancelled",
                            application_sme_df["LastModifiedDate"],
                        ).otherwise(lit(None)),
                    )
                    .withColumn("creationdate_b", application_sme_df["B.creationdate"])
                    .withColumn("InstallmentSN_b", application_sme_df["B.InstallmentSN"])
                    .withColumn("LBSParentID_b", application_sme_df["B.LBSParentID"])
                    .withColumn(
                        "IsPrimaryRowFlag",
                        when(col("B.InstallmentSN").isNull(), 1).otherwise(0),
                    )
                )

        # ================== application_sme_df ==================

        application_sme_primary_key_fields = [
                    business_plan_strategy.BusinessPlanID_strategy,
                    application_sme_df.InstallmentSN_b,
                    application_sme_df.id_strategy,
                    business_plan_rawdata.CollectionPlan_Month,
                    lit("sme"),
                ]

        applicationkey_sme_primary_key_fields = [
                    business_plan_strategy.BusinessPlanID_strategy,
                    application_sme_df.InstallmentSN_b,
                    application_sme_df.id_strategy,
                    business_plan_rawdata.CollectionPlan_Month,
                ]

        application_sme_df = application_sme_df.withColumn(
                    "ApplicationIdKey",
                    sha2(concat_ws(";", *application_sme_primary_key_fields), 256),
                ).withColumn(
                    "ApplicationKey",
                    concat_ws(";", *applicationkey_sme_primary_key_fields),
                )

        application_sme = application_sme_df.select(
                    application_sme_df.ApplicationIdKey.cast("String").alias("ApplicationId"),
                    lit("sme").alias("SourceSystem"),
                    portfolio.unit_code_profits.cast("string").alias("PortfolioKey"),
                    portfolio.PortfolioName.cast("String").alias("PortfolioName"),
                    lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias("IngestionTimestamp"),
                    lit(None).alias("BatchID"),
                    application_sme_df.ApplicationKey.cast("String").alias("ApplicationKey"),
                    application_sme_df.id_strategy.cast("String").alias("ApplicationStrategyKey"),
                    application_sme_df.LBSParentID_b.cast("String").alias("ApplicationScheduleKey"),
                    lit(None).alias("ExposureKey"),
                    application_sme_df.AgreementId.cast("String").alias("AgreementKey"),
                    business_plan.DebtorCDI.cast("String").alias("CDI"),
                    lit(None).alias("PartyKey"),
                    business_plan.DebtorAFM.cast("String").alias("TaxRegistrationNumber"),
                    business_plan.GroupCode.cast("String").alias("GroupCode"),
                    business_plan.ID.cast("Int").alias("BusinessPlanKey"),
                    business_plan.SolutionType.cast("String").alias("BusinessPlanSolutionTypeCode"),
                    business_plan.SolutionType.cast("String").alias("BusinessPlanSolutionType"),
                    business_plan_rawdata.BusinessPlan_Status.cast("String").alias("BusinessPlanSolutionStatusCode"),
                    business_plan_rawdata.BusinessPlan_Status.cast("String").alias("BusinessPlanSolutionStatus"),
                    business_plan.comments_bp.cast("String").alias("BusinessPlanComments"),

                    business_plan.creationdate_bp.cast("Timestamp").alias("BusinessPlanCreatedDate"),
                    business_plan.creationuser_bp.cast("String").alias("BusinessPlanCreatedBy"),
                    business_plan.LastmodifiedDate_bp.cast("Timestamp").alias("BusinessPlanLastUpdatedDate"),
                    business_plan.LastModifiedUser.cast("String").alias("BusinessPlanLastUpdatedBy"),
                    business_plan_strategy.Strategy_id.cast("String").alias("ApplicationTypeCode"),
                    business_plan_strategy.Strategy.cast("String").alias("ApplicationType"),
                    business_plan_strategy.bps_status.cast("String").alias("ApplicationStatusCode"),
                    business_plan_strategy.bps_status.cast("String").alias("ApplicationStatus"),
                    business_plan.ApplicationDate.cast("Timestamp").alias("ApplicationDate"),
                    business_plan.IntentionDate.cast("date").alias("IntentionDate"),
                    lit(None).alias("ApprovedDate"),
                    lit(None).alias("RejectedDate"),
                    application_sme_df.ApplicationCancelledDate.cast("Timestamp").alias("CancelledDate"),
                    business_plan.CompletionTargetDate.cast("String").alias("ImplementationDate"),
                    business_plan_strategy.BeginMonth.alias("BeginDate"),
                    business_plan_strategy.ProposedBy.alias("ProposedTypeCode"),
                    business_plan_strategy.ProposedBy.alias("ProposedType"),
                    business_plan_strategy.Confidence.alias("ConfidenceTypeCode"),
                    business_plan_strategy.Confidence.alias("ConfidenceType"),
                    lit(None).alias("Installment"),
                    lit(None).alias("PropositionUser"),
                    lit(None).alias("Tenor"),

                    business_plan_rawdata.TimeToAuction.cast("Int").alias("TimeToAuction"),
                    business_plan_rawdata.TimeToRecovery.cast("Int").alias("TimeToRecovery"),
                    business_plan_rawdata.TimeToSell.cast("Int").alias("TimeToSell"),
                    business_plan_rawdata.PrepaymentMonth.cast("Date").alias("PrepaymentDate"),
                    business_plan_rawdata.AuctionPrice.cast("Decimal(19,2)").alias("AuctionPrice"),
                    business_plan_rawdata.BalloonAmount.cast("Decimal(19,2)").alias("BalloonAmount"),
                    business_plan_rawdata.PrepaymentAmount.cast("Decimal(19,2)").alias("PrepaymentAmount"),
                    business_plan_rawdata.BusinessPlanStrategy_SolutionType.cast("String").alias("BusinessPlanStrategySolutionTypeCode"),
                    business_plan_rawdata.BusinessPlanStrategy_SolutionType.cast("String").alias("BusinessPlanStrategySolutionType"),
                    business_plan_rawdata.Interest.cast("Decimal(21,4)").alias("Interest"),
                    business_plan_rawdata.InterestPaymentPeriodicityID.cast("Int").alias("InterestPaymentPeriodicity"),
                    business_plan_rawdata.NetRecoveryAmount.cast("Decimal(19,2)").alias("NetRecoveryAmount"),
                    business_plan_rawdata.BP_Strategy_Code.cast("Int").alias("BusinessPlanStrategyCode"),
                    business_plan_strategy.Comments.cast("String").alias("Comments"),
                    business_plan_strategy.LastModifiedDate.cast("Timestamp").alias("LastUpdatedDate"),
                    application_sme_df.creationdate_b.cast("Timestamp").alias("CreatedDate"),
                    business_plan_strategy.CreationUser.cast("String").alias("CreatedBy"),
                    lit(None).cast("String").alias("AssigneeUserCode"),
                    lit(None).cast("String").alias("AssigneeUser"),
                    lit(None).cast("String").alias("AssigneePortfolioGroupCode"),
                    lit(None).cast("String").alias("AssigneePortfolioGroup"),
                    lit(None).cast("String").alias("IntegrationCode"),
                    lit(None).cast("String").alias("CommunicationChannelCode"),
                    lit(None).cast("String").alias("CommunicationChannel"),
                    lit(None).cast("String").alias("FlowSegmentCode"),
                    lit(None).cast("String").alias("FlowSegment"),
                    lit(None).cast("String").alias("UserCreatedQualcoGroupCode"),
                    lit(None).cast("String").alias("ContractSigningMethodCode"),
                    lit(None).cast("String").alias("ContractSigningMethod"),
                    lit(None).cast("String").alias("ManagedByPortfolioGroupCode"),
                    lit(None).cast("Boolean").alias("InterbankFlag"),
                    lit(None).cast("Boolean").alias("OcwFlag"),
                    lit(None).cast("Int").alias("ContactSigningPlaceTypeCode"),
                    lit(None).cast("String").alias("ContactSigningPlaceType"),
                    lit(None).cast("Int").alias("ReportingApplicationStatusCode"),
                    lit(None).cast("String").alias("ReportingApplicationStatus"),
                    lit(None).cast("Int").alias("ClientResponseCode"),
                    lit(None).cast("String").alias("ClientResponse"),
                    lit(None).cast("Int").alias("OcwSimulationVote"),
                    lit(None).cast("Timestamp").alias("StartedDate"),
                    lit(None).cast("Timestamp").alias("SubmittedDate"),
                    lit(None).cast("Timestamp").alias("QualcoAppPreApprovedDate"),
                    lit(None).cast("Timestamp").alias("QualcoAppDisbursedDate"),
                    lit(None).cast("Timestamp").alias("QualcoAppFinalStepDate"),
                    lit(None).cast("String").alias("UserCreatedChannelCode"),
                    lit(None).cast("String").alias("UserUnit"),
                    lit(None).cast("String").alias("UserCreatedGroupCode"),
                    lit(None).cast("String").alias("UserCreatedGroup"),

                    business_plan_rawdata.ImplementationStatus.cast("String").alias("ApplicationImplementationStatusCode"), # Changed from None 
                    business_plan_rawdata.ImplementationStatus.cast("String").alias("ApplicationImplementationStatus"), # Changed from None 

                    lit(None).alias("ApplicationActivationDate"),
                    lit(None).alias("ContactSigningPlaceCode"),
                    lit(None).alias("PolicyOverrideFlag"),
                    application_sme_df.InstallmentSN_b.cast("Int").alias("InstallmentNumber"),
                    business_plan_lbspaymentschedule.InstallmentTotalAmount.cast("Decimal(18,4)").alias("InstallmentAmount"),
                    business_plan_lbspaymentschedule.InstallmentDate.cast("Timestamp").alias("InstallmentDate"),
                    business_plan_lbspaymentschedule.InstallmentCapitalAmount.cast("Decimal(18,4)").alias("InstallmentCapitalAmount"),
                    business_plan_lbspaymentschedule.InstallmentChargeAmount.cast("Decimal(18,4)").alias("InstallmentChargeAmount"),
                    business_plan_lbspaymentschedule.InstallmentN128Amount.cast("Decimal(18,4)").alias("InstallmentN128Amount"),
                    business_plan_lbspaymentschedule.InstallmentTotalChargeAmount.cast("Decimal(18,4)").alias("InstallmentTotalChargeAmount"),
                    business_plan_lbspaymentschedule.InstallmentRemainingCapitalAmount.cast("Decimal(18,4)").alias("InstallmentRemainingCapitalAmount"),
                    business_plan_lbspaymentschedule.InstallmentNPVAmount.cast("Decimal(18,4)").alias("InstallmentNPVAmount"),
                    business_plan_lbspaymentschedule.InstallmentNPVCalculationDate.cast("Timestamp").alias("InstallmentNPVCalculationDate"),
                    business_plan_lbspaymentschedule.IsActive.cast("Boolean").alias("InstallementStatusFlag"),
                    application_sme_df.IsPrimaryRowFlag.cast("Int"),

                    business_plan_strategy.BusinessDate.cast("Date"),

                    # New columns added
                    business_plan_rawdata.CollectionPlan_Month.cast("Date").alias("CollectionPlanMonth"),
                    business_plan_rawdata.CollectionPlan_Amount.cast("Decimal(19,2)").alias("CollectionPlanAmount"),
                    business_plan_rawdata.tbl.cast("String").alias("SourceTableName"),

                )

        # ================== Append ==================

        application_sme.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)
    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
sme = ApplicationSMEInternal(spark, catalog_landed, run_date)

In [0]:
load_curated_application_sme(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix, sme)